[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/csv_auditing_llama_405b.ipynb)

# Human-Level CSV Auditing with Llama-3.1-405B-Instruct

**Open-source stack for nuance-heavy CSV evaluation at scale**

This notebook implements a production-grade auditing system for 5,000-row CSVs using:
- **Core Model**: Llama-3.1-405B-Instruct (GPT-4-class reasoning, 128k context)
- **Vector DB**: ChromaDB with BGE-M3 embeddings for reference retrieval
- **Output**: Structured JSON with correctness scores and calibration curves

## Hardware Requirements
- **Recommended**: 8×A100-80GB (full 405B model)
- **Alternative**: 2×RTX-6000-Ada-48GB (70B model with chunking)
- **Budget**: Use API inference (Together AI, Fireworks, Groq)

## 1. Installation & Setup

In [ ]:
%%capture
# Core dependencies
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.38.0 accelerate>=0.27.0
!pip install bitsandbytes>=0.42.0  # For quantization
!pip install chromadb>=0.4.22 sentence-transformers>=2.3.0
!pip install pandas numpy seaborn matplotlib plotly
!pip install chardet>=5.0.0  # For CSV encoding detection
!pip install vllm  # Optional: faster inference
!pip install auto-gptq optimum  # For AWQ/GPTQ quantization

print("✓ All dependencies installed")

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

import seaborn as sns
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Model selection (choose based on available VRAM)
MODEL_OPTIONS = {
    "405b": "meta-llama/Meta-Llama-3.1-405B-Instruct",  # 8×A100-80GB
    "70b": "meta-llama/Meta-Llama-3.1-70B-Instruct",    # 2×RTX-6000-Ada
    "8b": "meta-llama/Meta-Llama-3.1-8B-Instruct",      # Single GPU
    "mixtral": "mistralai/Mixtral-8x22B-Instruct-v0.1", # 4×A100-80GB
    "qwen": "Qwen/Qwen2-72B-Instruct"                   # Alternative 70B
}

# Configuration
CONFIG = {
    # Model settings
    "model_name": MODEL_OPTIONS["70b"],  # Change to "405b" if you have 8×A100
    "use_4bit": True,  # AWQ/GPTQ quantization
    "max_context": 128000,  # Llama-3.1 supports 128k
    
    # Batch processing
    "batch_size": 150,  # Rows per prompt (adjust based on VRAM)
    "max_tokens_per_case": 1000,  # Max tokens per CSV row summary
    
    # Vector DB for reference retrieval
    "embedding_model": "BAAI/bge-m3",  # Best multilingual embeddings
    "top_k_references": 7,  # Number of reference examples to retrieve
    
    # Generation parameters
    "temperature": 0.1,  # Low temperature for consistency
    "max_new_tokens": 4096,  # Enough for 150 structured JSON responses
    
    # Paths
    "csv_path": "your_data.csv",  # Upload your 5000-row CSV
    "reference_path": "reference_rules.txt",  # Upload reference file
    "output_dir": "./results"
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("✓ Configuration loaded")

## 3. Load Models

In [ ]:
def load_llm(model_name: str, use_4bit: bool = True):
    """Load Llama-3.1 with 4-bit quantization for reduced VRAM."""
    print(f"Loading {model_name}...")
    
    if use_4bit:
        # 4-bit quantization: 405B → ~220GB, 70B → ~38GB
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            use_cache=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    print(f"✓ Model loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, tokenizer

# Load main model
model, tokenizer = load_llm(CONFIG["model_name"], CONFIG["use_4bit"])

In [ ]:
# Load embedding model for vector search
print(f"Loading embedding model: {CONFIG['embedding_model']}...")
embedding_model = SentenceTransformer(CONFIG['embedding_model'])
print("✓ Embedding model loaded")

## 4. Vector Database Setup (Reference Retrieval)

In [ ]:
# Initialize ChromaDB
chroma_client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="./chroma_db"
))

# Create collection for reference rules
collection = chroma_client.get_or_create_collection(
    name="reference_rules",
    metadata={"description": "Reference rules and examples for CSV auditing"}
)

print("✓ ChromaDB initialized")

In [ ]:
def index_reference_file(file_path: str):
    """Index reference rules/examples into vector DB."""
    print(f"Indexing {file_path}...")
    
    # Read reference file
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split into chunks (each rule/example)
    # Adjust splitting logic based on your file structure
    chunks = content.split('\n\n')  # Assuming double-newline separators
    chunks = [c.strip() for c in chunks if c.strip()]
    
    # Generate embeddings
    embeddings = embedding_model.encode(chunks, show_progress_bar=True)
    
    # Add to ChromaDB
    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[f"ref_{i}" for i in range(len(chunks))]
    )
    
    print(f"✓ Indexed {len(chunks)} reference chunks")
    return len(chunks)

# Upload your reference file and index it
# from google.colab import files
# uploaded = files.upload()  # Upload reference_rules.txt

# For now, create a sample reference file
sample_reference = """Rule 1: High-quality responses must be specific and detailed.
Example: "Good response provides exact numbers and citations."

Rule 2: Responses should address all parts of the question.
Example: "A complete answer covers What, Why, and How."

Rule 3: Avoid vague language like "might", "could", "possibly".
Example: "Use definitive language: 'This indicates...' not 'This might indicate...'"
"""

with open(CONFIG['reference_path'], 'w') as f:
    f.write(sample_reference)

num_indexed = index_reference_file(CONFIG['reference_path'])

In [ ]:
def retrieve_relevant_rules(query: str, top_k: int = 7) -> List[str]:
    """Retrieve most relevant reference rules for a query."""
    query_embedding = embedding_model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    
    return results['documents'][0] if results['documents'] else []

# Test retrieval
test_query = "How should responses handle complex questions?"
relevant_rules = retrieve_relevant_rules(test_query, top_k=3)
print("\nTest retrieval for:", test_query)
for i, rule in enumerate(relevant_rules, 1):
    print(f"\n{i}. {rule[:100]}...")

## 5. CSV Processing & Context Management

In [ ]:
import chardet
from io import StringIO
import csv

def detect_encoding(file_path: str) -> str:
    """Detect file encoding using chardet."""
    with open(file_path, 'rb') as f:
        raw_data = f.read(100000)  # Read first 100KB
    result = chardet.detect(raw_data)
    encoding = result['encoding']
    confidence = result['confidence']
    print(f"Detected encoding: {encoding} (confidence: {confidence:.2%})")
    return encoding if confidence > 0.7 else 'utf-8'

def detect_delimiter(file_path: str, encoding: str) -> str:
    """Detect CSV delimiter."""
    with open(file_path, 'r', encoding=encoding) as f:
        sample = f.read(4096)
    
    sniffer = csv.Sniffer()
    try:
        dialect = sniffer.sniff(sample)
        delimiter = dialect.delimiter
        print(f"Detected delimiter: '{delimiter}'")
        return delimiter
    except:
        # Fallback: count occurrences
        delimiters = [',', ';', '\t', '|']
        counts = {d: sample.count(d) for d in delimiters}
        delimiter = max(counts, key=counts.get)
        print(f"Detected delimiter (fallback): '{delimiter}'")
        return delimiter

def load_csv_robust(file_path: str) -> pd.DataFrame:
    """Robustly load CSV with automatic encoding and delimiter detection."""
    print(f"\nLoading CSV: {file_path}")
    print("=" * 50)
    
    # Try multiple encoding strategies
    encodings = [
        detect_encoding(file_path),  # Auto-detected
        'utf-8',
        'utf-8-sig',  # UTF-8 with BOM
        'latin1',
        'cp1252',  # Windows encoding
        'iso-8859-1',
        'ascii'
    ]
    
    df = None
    successful_encoding = None
    
    for encoding in encodings:
        try:
            print(f"\nTrying encoding: {encoding}")
            
            # Detect delimiter
            delimiter = detect_delimiter(file_path, encoding)
            
            # Try reading with pandas
            df = pd.read_csv(
                file_path,
                encoding=encoding,
                delimiter=delimiter,
                on_bad_lines='skip',  # Skip malformed lines
                engine='python',  # More flexible parser
                quoting=csv.QUOTE_MINIMAL,
                escapechar='\\',
                skipinitialspace=True
            )
            
            # Verify we got data
            if len(df) > 0 and len(df.columns) > 0:
                successful_encoding = encoding
                print(f"✓ Successfully loaded with {encoding}")
                break
                
        except Exception as e:
            print(f"✗ Failed with {encoding}: {str(e)[:100]}")
            continue
    
    if df is None:
        raise ValueError("Could not load CSV with any encoding. Please check file format.")
    
    # Clean column names (remove BOM, whitespace, etc.)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '')
    
    # Remove completely empty rows
    df = df.dropna(how='all')
    
    # Reset index
    df = df.reset_index(drop=True)
    
    print("\n" + "=" * 50)
    print(f"✓ CSV loaded successfully!")
    print(f"  Encoding: {successful_encoding}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Column names: {df.columns.tolist()}")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df

In [ ]:
# OPTION 1: Upload your CSV file
# Uncomment the following lines to upload from your computer:
# from google.colab import files
# uploaded = files.upload()
# csv_filename = list(uploaded.keys())[0]
# CONFIG['csv_path'] = csv_filename

# OPTION 2: Mount Google Drive
# Uncomment to load from Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# CONFIG['csv_path'] = '/content/drive/MyDrive/your_data.csv'

# OPTION 3: Use sample data for testing
print("Creating sample data for demonstration...\n")
sample_data = {
    'case_id': range(1, 101),
    'original_text': [f"Sample case {i}: Complex scenario with nuanced details requiring careful evaluation..." for i in range(1, 101)],
    'eval_text': [f"Evaluation for case {i}: Comprehensive assessment of quality, completeness, and accuracy..." for i in range(1, 101)]
}
df = pd.DataFrame(sample_data)
print(f"✓ Created sample dataset: {len(df)} rows, {len(df.columns)} columns")

# If you have a real CSV file, load it robustly:
# df = load_csv_robust(CONFIG['csv_path'])

# Display preview
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
def create_case_summary(row: pd.Series, max_tokens: int = 1000) -> str:
    """Convert CSV row into concise case summary (≤1k tokens)."""
    # Customize this based on your CSV structure
    summary = f"""Case #{row.get('case_id', 'N/A')}:
Original: {str(row.get('original_text', ''))[:500]}
Evaluation: {str(row.get('eval_text', ''))[:500]}
"""
    
    # Truncate to max_tokens (rough estimate: 1 token ≈ 4 chars)
    max_chars = max_tokens * 4
    return summary[:max_chars]

# Create summaries for all rows
df['case_summary'] = df.apply(
    lambda row: create_case_summary(row, CONFIG['max_tokens_per_case']), 
    axis=1
)

print(f"✓ Created {len(df)} case summaries")
print("\nSample summary:")
print(df['case_summary'].iloc[0])

## 6. Prompt Engineering with Retrieved Context

In [ ]:
def build_batch_prompt(batch_df: pd.DataFrame, retrieved_rules: List[str]) -> str:
    """Build structured prompt for batch of CSV rows with retrieved rules."""
    
    # Context: Retrieved reference rules
    context_section = """# EVALUATION GUIDELINES (Retrieved from Reference Database)\n"""
    for i, rule in enumerate(retrieved_rules, 1):
        context_section += f"\n## Guideline {i}\n{rule}\n"
    
    # Cases to evaluate
    cases_section = "\n# CASES TO EVALUATE\n\n"
    for idx, row in batch_df.iterrows():
        cases_section += f"{row['case_summary']}\n---\n"
    
    # Instruction for structured output
    instruction = f"""# TASK
Evaluate each case above using the provided guidelines. For each case, provide:
1. A correctness score (0-100)
2. A brief reason (max 50 words)

Return your response as a JSON array with this structure:
[
  {{"row_id": 1, "score": 85, "reason": "Clear and specific but missing one key detail..."}},
  {{"row_id": 2, "score": 92, "reason": "Comprehensive response addressing all aspects..."}}
]

Respond ONLY with the JSON array, no additional text.
"""
    
    full_prompt = context_section + cases_section + instruction
    return full_prompt

# Test prompt building
sample_batch = df.head(3)
sample_rules = retrieve_relevant_rules("evaluate case quality", top_k=5)
test_prompt = build_batch_prompt(sample_batch, sample_rules)

print(f"Prompt length: {len(test_prompt)} chars")
print(f"Est. tokens: {len(test_prompt) // 4}")
print("\nPrompt preview (first 500 chars):")
print(test_prompt[:500])

## 7. Batch Inference Pipeline

In [ ]:
def generate_evaluation(prompt: str) -> str:
    """Generate evaluation using Llama-3.1."""
    messages = [
        {"role": "system", "content": "You are an expert evaluator. Provide structured JSON responses."},
        {"role": "user", "content": prompt}
    ]
    
    # Format with chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(
        formatted_prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=CONFIG['max_context']
    ).to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG['max_new_tokens'],
            temperature=CONFIG['temperature'],
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    if "assistant" in generated_text:
        response = generated_text.split("assistant")[-1].strip()
    else:
        response = generated_text
    
    return response

# Test generation
print("Testing generation on sample batch...")
test_output = generate_evaluation(test_prompt)
print("\nGenerated output:")
print(test_output[:500])

In [ ]:
def parse_json_response(response: str) -> List[Dict]:
    """Extract JSON from model response."""
    try:
        # Try direct parsing
        return json.loads(response)
    except:
        # Extract JSON from markdown code blocks
        import re
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
        if json_match:
            return json.loads(json_match.group(1))
        
        # Try finding [...] array
        array_match = re.search(r'\[\s*\{[\s\S]*?\}\s*\]', response)
        if array_match:
            return json.loads(array_match.group(0))
        
        raise ValueError("Could not extract JSON from response")

# Test parsing
try:
    parsed = parse_json_response(test_output)
    print(f"✓ Successfully parsed {len(parsed)} evaluations")
    print(f"Sample: {parsed[0] if parsed else 'None'}")
except Exception as e:
    print(f"⚠ Parsing failed: {e}")

In [ ]:
def process_all_rows(df: pd.DataFrame, batch_size: int) -> List[Dict]:
    """Process entire CSV in batches."""
    all_results = []
    num_batches = (len(df) + batch_size - 1) // batch_size
    
    print(f"Processing {len(df)} rows in {num_batches} batches...\n")
    
    for i in range(0, len(df), batch_size):
        batch_num = i // batch_size + 1
        batch_df = df.iloc[i:i + batch_size].copy()
        
        print(f"Batch {batch_num}/{num_batches}: Rows {i+1}-{min(i+batch_size, len(df))}")
        
        # Retrieve relevant rules for this batch
        # Use first case as query (or aggregate all cases)
        batch_query = batch_df['case_summary'].iloc[0]
        relevant_rules = retrieve_relevant_rules(
            batch_query, 
            top_k=CONFIG['top_k_references']
        )
        
        # Build prompt
        prompt = build_batch_prompt(batch_df, relevant_rules)
        
        # Generate
        response = generate_evaluation(prompt)
        
        # Parse
        try:
            batch_results = parse_json_response(response)
            all_results.extend(batch_results)
            print(f"  ✓ Evaluated {len(batch_results)} rows")
        except Exception as e:
            print(f"  ⚠ Error parsing batch {batch_num}: {e}")
            # Store raw response for debugging
            for idx in batch_df.index:
                all_results.append({
                    "row_id": idx,
                    "score": None,
                    "reason": "Parse error",
                    "raw_response": response[:200]
                })
        
        print()
    
    return all_results

# Run full evaluation (use smaller subset for demo)
print("Starting evaluation...\n")
results = process_all_rows(
    df.head(20),  # Process first 20 rows for demo
    batch_size=5
)

print(f"\n✓ Completed evaluation of {len(results)} rows")

## 8. Results Analysis & Visualization

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Merge with original data
df_final = df.merge(
    results_df, 
    left_index=True, 
    right_on='row_id', 
    how='left'
)

# Calculate correctness probability
df_final['correctness_prob'] = df_final['score'] / 100

# Save results
output_file = f"{CONFIG['output_dir']}/evaluation_results.csv"
df_final.to_csv(output_file, index=False)
print(f"✓ Results saved to {output_file}")

# Display summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"Total rows evaluated: {len(df_final)}")
print(f"Mean score: {df_final['score'].mean():.2f}")
print(f"Median score: {df_final['score'].median():.2f}")
print(f"Std deviation: {df_final['score'].std():.2f}")
print(f"Min score: {df_final['score'].min():.2f}")
print(f"Max score: {df_final['score'].max():.2f}")

df_final.head()

In [ ]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Score distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df_final['score'].dropna(), bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(df_final['score'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df_final["score"].mean():.1f}')
axes[0].set_xlabel('Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Score Distribution', fontsize=14, fontweight='bold')
axes[0].legend()

# Box plot
axes[1].boxplot(df_final['score'].dropna())
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Score Box Plot', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/score_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualizations saved")

In [ ]:
# Calibration curve (if you have ground truth labels)
# This assumes you have a 'ground_truth_score' column
# For demo purposes, we'll skip this or use synthetic data

# Example calibration plot
fig, ax = plt.subplots(figsize=(8, 8))

# If you have ground truth:
# from sklearn.calibration import calibration_curve
# prob_true, prob_pred = calibration_curve(
#     df_final['ground_truth_binary'], 
#     df_final['correctness_prob'], 
#     n_bins=10
# )
# ax.plot(prob_pred, prob_true, marker='o', label='Model')

# Perfect calibration line
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('True Probability', fontsize=12)
ax.set_title('Calibration Curve', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.savefig(f"{CONFIG['output_dir']}/calibration_curve.png", dpi=300, bbox_inches='tight')
plt.show()

## 9. Failure Mode Analysis with LLM

In [ ]:
def generate_failure_summary(low_score_cases: pd.DataFrame) -> str:
    """Use LLM to summarize common failure modes."""
    
    # Compile low-scoring cases
    cases_text = "\n\n".join([
        f"Case {row['case_id']}: Score {row['score']:.0f} - Reason: {row['reason']}"
        for _, row in low_score_cases.iterrows()
    ])
    
    summary_prompt = f"""Analyze the following low-scoring evaluation cases and identify common failure patterns.

{cases_text}

Provide a concise summary (≤300 words) of:
1. Most common failure modes
2. Root causes
3. Recommendations for improvement
"""
    
    messages = [
        {"role": "system", "content": "You are an expert at analyzing evaluation patterns."},
        {"role": "user", "content": summary_prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.3,
            do_sample=True
        )
    
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in summary:
        summary = summary.split("assistant")[-1].strip()
    
    return summary

# Generate failure analysis for bottom 20% of scores
threshold = df_final['score'].quantile(0.20)
low_score_cases = df_final[df_final['score'] <= threshold].head(10)

print(f"Analyzing {len(low_score_cases)} low-scoring cases (threshold: {threshold:.1f})...\n")
failure_summary = generate_failure_summary(low_score_cases)

print("=== FAILURE MODE ANALYSIS ===")
print(failure_summary)

# Save summary
with open(f"{CONFIG['output_dir']}/failure_analysis.txt", 'w') as f:
    f.write(failure_summary)

print(f"\n✓ Analysis saved to {CONFIG['output_dir']}/failure_analysis.txt")

## 10. Export & Next Steps

In [ ]:
# Create comprehensive report
report = f"""# CSV Auditing Report
Generated: {pd.Timestamp.now()}

## Configuration
- Model: {CONFIG['model_name']}
- Total rows: {len(df_final)}
- Batch size: {CONFIG['batch_size']}
- Reference chunks indexed: {num_indexed}

## Performance Metrics
- Mean score: {df_final['score'].mean():.2f}
- Median score: {df_final['score'].median():.2f}
- Standard deviation: {df_final['score'].std():.2f}
- Range: {df_final['score'].min():.2f} - {df_final['score'].max():.2f}

## Score Distribution
- Top 25%: > {df_final['score'].quantile(0.75):.2f}
- Middle 50%: {df_final['score'].quantile(0.25):.2f} - {df_final['score'].quantile(0.75):.2f}
- Bottom 25%: < {df_final['score'].quantile(0.25):.2f}

## Files Generated
1. evaluation_results.csv - Full results with scores and reasons
2. score_distribution.png - Histogram and box plot
3. calibration_curve.png - Model calibration analysis
4. failure_analysis.txt - Common failure patterns

## Next Steps
1. Review low-scoring cases for pattern analysis
2. Refine reference rules based on failure modes
3. Consider ensemble with multiple models for consensus scoring
4. Implement active learning for borderline cases
"""

with open(f"{CONFIG['output_dir']}/REPORT.md", 'w') as f:
    f.write(report)

print(report)
print(f"\n✓ Full report saved to {CONFIG['output_dir']}/REPORT.md")

In [ ]:
# Download results (in Colab)
# from google.colab import files
# import shutil

# # Create zip of all results
# shutil.make_archive('results', 'zip', CONFIG['output_dir'])
# files.download('results.zip')

print("✓ All processing complete!")
print(f"\nResults directory: {CONFIG['output_dir']}")
print("Files generated:")
for file in Path(CONFIG['output_dir']).glob('*'):
    print(f"  - {file.name}")

## Alternative: API-Based Inference (No GPU Required)

If you don't have access to 8×A100 or want faster/cheaper inference, use API providers:

In [ ]:
# Alternative: Use Together AI, Fireworks, or Groq API
# Much cheaper than renting 8×A100!

# Example with Together AI:
"""
import together

together.api_key = "your-api-key"

response = together.Complete.create(
    model="meta-llama/Meta-Llama-3.1-405B-Instruct-Turbo",
    prompt=your_prompt,
    max_tokens=4096,
    temperature=0.1
)

# Cost: ~$3.50 per 1M input tokens, ~$4.50 per 1M output tokens
# For 5000 rows × 1k tokens each + outputs ≈ $20-30 total
"""

print("See comments above for API-based inference option")